# NLP-07 · Instruction Tuning and LoRA
### From next-token completion to response-supervised behavior

NLP-03 produced a domain-adapted checkpoint. Now we format instruction-response pairs,
mask prompt labels, perform real SFT, and compare full parameter updates with rank-4
LoRA adapters. **Prerequisites:** NLP-03, DL-03, and DL-04.


> **Canonical learner route · module NLP-07 of 65**
>
> Required prerequisites: **NLP-03, DL-03, DL-04** · Previous: **NLP-03** ·
> Next after mastery: **NLP-08** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives
- Construct shifted inputs and response-only labels.
- Explain how prompts condition responses without direct prompt loss.
- Train full-SFT and LoRA candidates from the same checkpoint.
- Trace LoRA shapes, initialization, trainable parameters, merging, and limitations.
- Compare training, held-out, and retention evidence before selecting an adapter.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · NLP-07 · Instruction Tuning and LoRA

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation
Raw pretrained models continue text; supervised instruction tuning demonstrates the
desired interaction. Full tuning updates every parameter and optimizer state. LoRA
instead learns low-rank corrections to selected linear projections, making adaptation
state smaller—but not guaranteeing equal quality or preserved behavior.


## 3 · Intuition and Practical Problem
A prompt is the question sheet; the response is the portion graded directly. The model
must read the question, but we need not reward it for reproducing the question. LoRA is
a removable correction layer over a frozen base. This analogy does not imply adapters
store isolated human-readable skills.

Use SFT for stable formats and demonstrated behavior. Try prompting first for small
changes and RAG for changing facts. Avoid tuning without representative held-out and
retention sets.


## 4 · Mathematical Foundations
Read aloud: “average negative log probability only over response positions.”

$$J_{SFT}=-\frac{1}{\sum_t m_t}\sum_t m_t\log p_\theta(x_{t+1}\mid x_{1:t}).$$

$m_t\in\{0,1\}$ is the response-label mask; all other symbols match NLP-03. With losses
`[2.0,1.0,0.4,0.2]` and mask `[0,0,1,1]`, $J=(0.4+0.2)/2=0.3$.

For PyTorch's column-vector convention, a frozen linear layer becomes
$h=W_0x+(\alpha/r)BAx$. $W_0\in\mathbb R^{d_{out}\times d_{in}}$ is frozen,
$A\in\mathbb R^{r\times d_{in}}$, $B\in\mathbb R^{d_{out}\times r}$, rank $r$ is
small, and $\alpha/r$ scales the update. Initializing $B=0$ makes the initial correction
exactly zero. LoRA stores $r(d_{in}+d_{out})$ trainable weights instead of
$d_{in}d_{out}$ for one projection.


## 5 · Manual Implementation from Scratch
The project builds each `user/assistant` sequence, shifts targets one character, assigns
`-100` to prompt and padding labels, and uses PyTorch cross-entropy's ignore index.
Full and LoRA models start from identical continued-pretraining weights.


In [ ]:
import sys
from pathlib import Path
repo=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'projects/language_model_adaptation').exists())
sys.path[:0]=[str(repo/'projects/language_model_adaptation/src'),str(repo/'projects/tiny_language_model/src')]
from language_model_adaptation.lab import run_adaptation_lab
report=run_adaptation_lab(seed=42)
print(report['instruction_tuning'])


## 6 · Visualization


In [ ]:
import matplotlib.pyplot as plt
t=report['instruction_tuning']; names=['full','lora']
plt.bar([0,1],[t[n]['train_loss_after'] for n in names],label='train')
plt.bar([.3,1.3],[t[n]['held_out_loss'] for n in names],width=.3,label='held out')
plt.xticks([.15,1.15],names); plt.ylabel('response-token loss'); plt.legend(); plt.show()


## 7 · Failure Modes and Common Mistakes
| Symptom | Cause | Check | Repair |
|---|---|---|---|
| prompt dominates loss | labels not masked | supervised-token count | set prompt labels to `-100` |
| adapter changes output at start | nonzero B | zero-delta test | initialize B to zero |
| tiny train loss, weak held-out | memorization | validation loss | more diverse data/regularization |
| base behavior regresses | combined adapter changes outputs | retention suite | lower capacity/LR or replay |

Do not claim LoRA “prevents forgetting,” compare models with different starting
checkpoints, or treat parameter percentage as a quality metric.


## 8 · Library or Tool Implementation
PEFT libraries automate adapter insertion and saving. Verify target-module names,
fan-in/fan-out convention, bias policy, precision, and merged/unmerged equivalence.
The scratch implementation remains the behavioral oracle.


## 9 · Realistic Case Study
A support model needs a strict response schema. The team first tests structured prompts,
then SFTs reviewed examples. It compares full and LoRA tuning on format validity, task
correctness, old-task retention, latency, and state size—not training loss alone.


## 10 · Learning and Production Considerations
Chat templates and special tokens are part of the model contract. Pack examples without
allowing one response to attend across unintended boundaries. Version base weights,
adapter, tokenizer, template, split, and evaluation suite together.


## 11 · Tradeoff Analysis
| Approach | Trainable state | Strength | Weakness |
|---|---:|---|---|
| prompt | none | fastest baseline | limited adaptation |
| LoRA | small | portable and cheaper | rank/targets need evaluation |
| full SFT | all | maximum update capacity | memory and regression risk |
| RAG | no weight update | fresh sourced facts | retrieval complexity |


## 12 · Readiness Check
The full model reaches lower train loss, while LoRA uses `768` versus `16,928` trainable
parameters and has better held-out loss in this tiny run. That does not prove LoRA is
universally better; it exposes full-SFT overfitting on four examples.


## 13 · Teach-Back
1. Which target positions receive loss?
2. Why do response gradients still depend on prompt states?
3. Why is B initialized to zero?
4. What does rank control?
5. What evidence would justify full SFT over LoRA?


## 14 · Exercises, Self-Check, and Solutions
**Worked (10 min):** apply mask `[0,0,1,1]` to four losses; answer `0.3`.
**Guided (20 min):** print supervised character counts; hint: count labels not equal to
`-100`. **Independent (45 min):** compare ranks 1, 4, 8 on held-out and retention loss.
**Challenge (60 min):** verify merged and unmerged LoRA logits match within `1e-6`.

Summary: SFT teaches from response labels; LoRA changes how many parameters move, not
what evidence is required. **Memory aid:** *Mask the prompt, train the response, and
measure the adapter like any other model.*


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **NLP-07 · Instruction Tuning and LoRA** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember NLP-07 · Instruction Tuning and LoRA: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · NLP-07

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
